# 09 · Map-reduce with `Send`

**Goal:** by the end you can run one node once **per item** of a list, with each run getting its own input, and
combine all the results in a final node.

## The idea in plain English

In lesson 08 the parallel branches were fixed when we built the graph: always exactly four. Often you don't know how
many you'll need until the graph runs. Our input is now a **list** of expressions, such as `["1+2", "3*4", "10/5"]`,
and we want to evaluate each one.

**`Send`** means "run this node once, with this specific input". A conditional edge can return a **list** of `Send`
objects, one per item, and LangGraph runs them all in parallel in the same superstep. Each run sees only the input
it was sent, not the whole state. Their results are collected by a reducer, and a final node combines them.

This pattern is called **map-reduce**. *Map* is doing the same job on every item (evaluate each expression).
*Reduce* is combining the answers into one (add them up).

**Analogy:** a teacher hands each student a **different** question card. Each student only sees their own card. The
teacher collects the answers and adds up the total.

## Picture

The graph has **one** `evaluate` node, but it runs once per expression. The dashed arrow carries a `Send` for each.

```mermaid
flowchart LR
    S([START]) -. "Send #quot;1+2#quot;" .-> E1["evaluate"]
    S -. "Send #quot;3*4#quot;" .-> E2["evaluate"]
    S -. "Send #quot;10/5#quot;" .-> E3["evaluate"]
    E1 --> T[total]
    E2 --> T
    E3 --> T
    T --> X([END])
```

## Step 1: Two state schemas

`MapReduceState` is the graph's state: the list of expressions, the `results` collected from every run, and the
`total`. `results` uses the `operator.add` reducer, because several `evaluate` runs write to it in the same
superstep (lesson 08).

`EvalState` is the input of **one** `evaluate` run: just one expression.

In [1]:
import operator
from typing import Annotated, TypedDict

from langgraph.graph import StateGraph, START, END


class MapReduceState(TypedDict):
    exprs: list[str]
    results: Annotated[list[float], operator.add]
    total: float


class EvalState(TypedDict):
    expr: str  # one expression, e.g. "3*4"

## Step 2: The `evaluate` node

These expressions have no spaces, so `evaluate` looks for each operator symbol in turn and splits the text there.
`OPERATIONS` maps each symbol to the matching function from the `operator` module (`operator.mul(3, 4)` is `12`).

Why not Python's built-in `eval("3*4")`? Because `eval` runs *any* Python code it's given. Never use it on text that
comes from a user.

In [2]:
OPERATIONS = {"+": operator.add, "-": operator.sub, "*": operator.mul, "/": operator.truediv}


def evaluate(state: EvalState) -> dict:
    expr = state["expr"]
    for symbol, fn in OPERATIONS.items():
        if symbol in expr:
            left, right = expr.split(symbol)
            value = fn(float(left), float(right))
            print(f"  evaluate({expr!r}) = {value:g}")
            return {"results": [value]}
    raise ValueError(f"No operator found in {expr!r}")


print(evaluate({"expr": "3*4"}))

  evaluate('3*4') = 12
{'results': [12.0]}


## Step 3: What a `Send` is

`Send(node, input)` is a small object holding two things: which node to run, and the input to give it. Making one
doesn't run anything. It's an instruction that LangGraph carries out.

In [3]:
from langgraph.types import Send

packet = Send("evaluate", {"expr": "1+2"})
print(packet)
print("node:", packet.node, "| input:", packet.arg)

Send(node='evaluate', arg={'expr': '1+2'})
node: evaluate | input: {'expr': '1+2'}


## Step 4: The fan-out function

`fan_out` is a router, like in lesson 05, but instead of returning one node name it returns a **list** of `Send`s,
one per expression. We call it directly to see the list.

In [4]:
def fan_out(state: MapReduceState) -> list[Send]:
    return [Send("evaluate", {"expr": expr}) for expr in state["exprs"]]


for packet in fan_out({"exprs": ["1+2", "3*4", "10/5"]}):
    print(packet)

Send(node='evaluate', arg={'expr': '1+2'})
Send(node='evaluate', arg={'expr': '3*4'})
Send(node='evaluate', arg={'expr': '10/5'})


## Step 5: The reduce node

`total` runs after all the `evaluate` runs, when `results` holds every value. It adds them up.

In [5]:
def total(state: MapReduceState) -> dict:
    print("  total of", state["results"])
    return {"total": sum(state["results"])}

## Step 6: Build the graph

`add_conditional_edges(START, fan_out, ["evaluate"])` runs `fan_out` as soon as the graph starts. The third argument
is a `path_map` again, this time as a **list** of the nodes the router might send to. A `Literal` hint can't describe
a list of `Send`s, so the `path_map` is how LangGraph learns about the dashed arrow for the diagram.

In [6]:
builder = StateGraph(MapReduceState)
builder.add_node("evaluate", evaluate)
builder.add_node("total", total)
builder.add_conditional_edges(START, fan_out, ["evaluate"])
builder.add_edge("evaluate", "total")
builder.add_edge("total", END)

graph = builder.compile()
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	evaluate(evaluate)
	total(total)
	__end__([<p>__end__</p>]):::last
	__start__ -.-> evaluate;
	evaluate --> total;
	total --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## Step 7: Run it

Three expressions, so `evaluate` runs three times, then `total` runs once.

In [7]:
final = graph.invoke({"exprs": ["1+2", "3*4", "10/5"]})
print(final)

  evaluate('1+2') = 3
  evaluate('3*4') = 12
  evaluate('10/5') = 2
  total of [3.0, 12.0, 2.0]
{'exprs': ['1+2', '3*4', '10/5'], 'results': [3.0, 12.0, 2.0], 'total': 17.0}


The graph didn't change, but the number of runs follows the input. Six expressions, six `evaluate` runs:

In [8]:
final = graph.invoke({"exprs": ["1+1", "2*3", "9-4", "8/2", "0.5*4", "100-1"]})
print("results:", final["results"])
print("total:  ", final["total"])

  evaluate('1+1') = 2
  evaluate('2*3') = 6
  evaluate('9-4') = 5
  evaluate('8/2') = 4
  evaluate('0.5*4') = 2
  evaluate('100-1') = 99
  total of [2.0, 6.0, 5.0, 4.0, 2.0, 99.0]
results: [2.0, 6.0, 5.0, 4.0, 2.0, 99.0]
total:   118.0


## Step 8: Each run sees only what it was sent

This is the big difference from a normal edge. A node reached by `Send` doesn't get the graph's state. It gets only
the dict inside the `Send`. To prove it, here is a node that prints exactly what it receives.

In [9]:
def peek(state: dict) -> dict:
    print("  peek received:", state)
    return {}


def fan_out_to_peek(state: MapReduceState) -> list[Send]:
    return [Send("peek", {"expr": expr}) for expr in state["exprs"]]


builder = StateGraph(MapReduceState)
builder.add_node("peek", peek)
builder.add_conditional_edges(START, fan_out_to_peek, ["peek"])
builder.add_edge("peek", END)
peeker = builder.compile()
peeker.invoke({"exprs": ["1+2", "3*4"]});  # the ; hides the returned state

  peek received: {'expr': '1+2'}
  peek received: {'expr': '3*4'}


There's no `exprs` and no `results` in what `peek` received. If a node needs more information, put it in the `Send`
payload.

## Step 9: An empty list

With no expressions, `fan_out` returns an empty list, so `evaluate` never runs. Since nothing runs `evaluate`,
nothing leads to `total` either, and the final state has **no** `total` at all.

In [10]:
print(graph.invoke({"exprs": []}))

{'exprs': [], 'results': []}


One fix: let the router go straight to `total` when there's nothing to evaluate. A router may return either a list of
`Send`s or a node name, as long as every destination is in the `path_map`.

In [11]:
def fan_out_safe(state: MapReduceState) -> list[Send] | str:
    if not state["exprs"]:
        return "total"
    return [Send("evaluate", {"expr": expr}) for expr in state["exprs"]]


builder = StateGraph(MapReduceState)
builder.add_node("evaluate", evaluate)
builder.add_node("total", total)
builder.add_conditional_edges(START, fan_out_safe, ["evaluate", "total"])
builder.add_edge("evaluate", "total")
builder.add_edge("total", END)
safe = builder.compile()

print(safe.invoke({"exprs": []}))
print(safe.invoke({"exprs": ["2*21"]}))

  total of []
{'exprs': [], 'results': [], 'total': 0}
  evaluate('2*21') = 42
  total of [42.0]
{'exprs': ['2*21'], 'results': [42.0], 'total': 42.0}


## What just happened

- **Step 2** tested `evaluate` on its own: `{'expr': '3*4'}` gave `{'results': [12.0]}`.
- **Step 4** showed `fan_out` turning three expressions into three `Send('evaluate', {...})` objects.
- **Step 7** ran `evaluate` three times in parallel, collected `[3.0, 12.0, 2.0]` with the reducer, and `total`
  added them to `17.0`. With six expressions, the same graph ran `evaluate` six times.
- **Step 8** showed that a node reached by `Send` gets only its payload, `{'expr': '1+2'}`, not the graph's state.
- **Step 9** showed that an empty list skips `total` completely, and fixed it by routing straight to `total`.

## Common mistakes

1. **No reducer on the results key.** It works with **one** item, then fails as soon as there are two:
   `InvalidUpdateError: At key 'results': Can receive only one value per step. Use an Annotated key to handle
   multiple values.`
   Fix: `results: Annotated[list[float], operator.add]`.
2. **A typo in the node name inside `Send`.** `Send("evalute", ...)` gives **no error**. LangGraph only logs
   `Ignoring unknown node name evalute in pending sends`, and `results` comes back empty.
   Fix: check the name, and list it in the `path_map` so the diagram shows the arrow.
3. **Expecting the sent node to see the whole state.** If `evaluate` reads `state["exprs"]`, it fails with
   `KeyError: 'exprs'`, because it only received `{"expr": ...}`.
   Fix: put everything the node needs into the `Send` payload.

## Try it

1. Add a key `largest: float` to the state, and make `total` also store the largest result. For
   `["1+2", "3*4", "10/5"]` it should be `12.0`.
2. Send each expression's **position** too: `Send("evaluate", {"expr": expr, "position": i})`, using
   `enumerate`. Make `evaluate` return lines like `"#2: 3*4 = 12"` into a `lines: Annotated[list[str], operator.add]`
   key. Then print the lines sorted by position. (Remember from lesson 08: the order of parallel results isn't
   guaranteed.)

Answers are in [`solutions/exercises_solutions.ipynb`](../solutions/exercises_solutions.ipynb).

## Key terms

New terms from this lesson, also in [GLOSSARY.md](../GLOSSARY.md):

- **`Send`**: `Send(node, input)` means "run this node once, with this input". A router can return a list of them.
- **Payload**: the input dict inside a `Send`. It's all the sent node receives.
- **Map-reduce**: run the same step on every item (map), then combine the answers (reduce).